# Training: Segment Length Scan (T ∈ {0.5, 1, 2, 3} s)

**Scope:**

- **Subject:** 2
- **Channel:** Fz
- **Time windows:** T ∈ {0.5, 1, 2, 3} s
- **Loss:** MSE (fixed)
- **CV:** GroupKFold, 5 folds over 40 trials
- **Train overlap:** 0% | **Test overlap:** 0%

**Settings (Section 4.3):**

- 20 epochs, batch size 16
- Adam: lr = 1e-4, weight_decay = 1e-5
- LR decayed by 0.1 every 10 epochs
- Identical network and training protocol as the loss comparison

**Runs:**

- T = 0.5 s — exp start 141
- T = 1.0 s — exp start 131
- T = 2.0 s — exp start 146
- T = 3.0 s — exp start 151

**Output directory:** `Segment-Lengths/Results/`

**Model directory:** `Segment-Lengths/Models/`

In [ ]:
# ============================================================
# 0. Environment setup (Colab or local)
# ============================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    !pip install -q pims


# ============================================================
# 1. Standard library
# ============================================================
import os
import sys
import pickle
import traceback

# ============================================================
# 2. Third-party
# ============================================================
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# 3. Project paths
# ============================================================
if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT   = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT   = r'D:\Thesis\Codes\Data'

# sanity check
if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under SOURCE_ROOT = {SOURCE_ROOT}. "
        f"Expected directory: {os.path.join(SOURCE_ROOT, 'face2eeg')}"
    )

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)


# ============================================================
# 4. Package imports
# ============================================================
from face2eeg.core.experiments import run_kfold_pipeline
from face2eeg.core.models      import try_gpu

# Framework constants (from config)
from face2eeg.core.config import (
    SEED,
    FS_EEG,
    FS_VIDEO,
)


# ============================================================
# 5. Reproducibility
# ============================================================
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# Configuration
# ============================================================
BASE = os.path.join(DATA_ROOT, 'Experiments', 'Cross-Validation')

EXP_DIR     = os.path.join(BASE, 'Segment-Lengths')
RESULTS_DIR = os.path.join(EXP_DIR, 'Results')
MODELS_DIR  = os.path.join(EXP_DIR, 'Models')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)


# ============================================================
# Experiment scope
# ============================================================
SUBJECT = 2
CHANNEL = 'Fz'


# ============================================================
# Segment lengths 
# ============================================================
# (variant label, time_window, exp_start)
RUNS = [
    ('tw0.5', 0.5, 141),
  # ('tw1',   1.0, 131), not included here for training again, because we trained the mse model on subject 2 with tw = 1 in the loss-function experiment
    ('tw2',   2.0, 151),
    ('tw3',   3.0, 156),
]


# ============================================================
# Hyperparameters 
# ============================================================
FRAME_SIZE   = (32, 32)

EPOCHS       = 20
BATCH_SIZE   = 16
LR           = 1e-4
WEIGHT_DECAY = 1e-5
SCHED_STEP   = 10
SCHED_GAMMA  = 0.1

# Data generation (no overlap for this experiment)
TRAIN_OVERLAP       = False
TRAIN_OVERLAP_RATIO = 0.0
TEST_OVERLAP        = False
TEST_OVERLAP_RATIO  = 0.0


# ============================================================
# Device
# ============================================================
device = try_gpu()


# ============================================================
# Sanity check
# ============================================================
print("EXP_DIR     :", EXP_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print("MODELS_DIR  :", MODELS_DIR)
print()
print("SUBJECT     :", SUBJECT)
print("CHANNEL     :", CHANNEL)
print("FRAME_SIZE  :", FRAME_SIZE)
print()
print("RUNS (variant, T, exp_start):")
for variant, tw, exp in RUNS:
    print(f"  {variant:<6}  T = {tw} s   exp_start = {exp}")
print()
print("EPOCHS      :", EPOCHS)
print("BATCH_SIZE  :", BATCH_SIZE)
print("LR          :", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("SCHED_STEP  :", SCHED_STEP)
print("SCHED_GAMMA :", SCHED_GAMMA)
print()
print("FS_EEG      :", FS_EEG, " (from config)")
print("FS_VIDEO    :", FS_VIDEO, " (from config)")
print("device      :", device)

## Run — MSE for each segment length

Loops over the four time-window values. Each run:
- Uses the same 5-fold GroupKFold splits
- Trains a fresh PhysNet-3DCNN-ED with MSE loss
- Saves results under `Results/sub02_chFz_tw{X}.pkl`
- Skips automatically if the file already exists

In [ ]:
for variant, tw, exp_start in RUNS:
    result_path = os.path.join(
        RESULTS_DIR, f'sub{SUBJECT:02d}_ch{CHANNEL}_{variant}.pkl'
    )

    if os.path.isfile(result_path):
        print(f"[skip] {result_path} already exists")
        continue

    print(f"\n{'#'*70}")
    print(f"#  {variant}  —  Subject {SUBJECT} | Channel {CHANNEL} | T = {tw} s")
    print(f"{'#'*70}")

    try:
        results = run_kfold_pipeline(
            subject_id=SUBJECT,
            channel_name=CHANNEL,
            time_window=tw,

            # CV splits
            n_points=40, n_splits=5,

            # scaler
            scaler_method='z-score',

            # data generation
            frame_size=FRAME_SIZE,
            video_fr=FS_VIDEO,
            signal_fs=FS_EEG,
            y_type='RawEEG',

            # train
            train_overlap=TRAIN_OVERLAP,
            train_overlap_ratio=TRAIN_OVERLAP_RATIO,
            train_shuffle_output=True,
            train_signal_normalize=True,

            # test
            test_overlap=TEST_OVERLAP,
            test_overlap_ratio=TEST_OVERLAP_RATIO,
            test_shuffle_output=False,
            test_signal_normalize=True,

            # model
            video_length=None,
            signal_length=None,

            # training
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            loss_fn=nn.MSELoss(),
            scheduler_step_size=SCHED_STEP,
            scheduler_gamma=SCHED_GAMMA,

            # prediction
            predict_batch_size=BATCH_SIZE,

            # output
            model_exp_start=exp_start,
            save_dir=MODELS_DIR,
            device=device,
            return_history=True,
            verbose=True,
            root_dir=DATA_ROOT,
        )

        with open(result_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"\n[saved] {result_path}")

    except Exception as e:
        print(f"\n!! ERROR for {variant} (T = {tw} s): {e}")
        traceback.print_exc()
        continue